# Feature Engineering Training v2

Jalankan setelah `Feature Engineering Combination.ipynb`. Semua pencarian memakai official training saja: 40.000 development training dan 10.000 validation. Screening memakai subset stratified 12.000/3.000. Setiap kandidat GridSearchCV disimpan ke ledger JSON sehingga eksekusi dapat dilanjutkan. Setelah konfigurasi dipilih dengan validation, SVM final di-fit pada seluruh 50.000 fitur training. Tidak ada pelatihan ulang CNN atau akses official test. Jalankan cell berurutan.

## 1. Konfigurasi

Grid awal menguji C=1/10/30 dan gamma=scale/1/3 bersama tiga profil bobot. B diprioritaskan, sedangkan A dan C tetap dipertimbangkan. Empat jam adalah target perencanaan, bukan jaminan runtime.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import time
from datetime import datetime, timezone

import joblib
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, PredefinedSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import Normalizer, StandardScaler
from sklearn.svm import SVC

SEED = 30092026
SOURCE_RUN = "cifar10_7fitur_v1"
RUN_ID = "cifar10_7fitur_v2"
ROOT = Path.cwd().resolve()
OUT = ROOT / "outputs" / RUN_ID / "fusion"
MANIFEST_PATH = OUT / "laporan" / "combination_manifest.json"
LEDGER_PATH = OUT / "laporan" / "search_ledger.json"
BUDGET_SECONDS = 4 * 3600
SCREEN_BUDGET_SECONDS = 40 * 60
FULL_BUDGET_SECONDS = 100 * 60
REFINE_BUDGET_SECONDS = 20 * 60
FINAL_RESERVE_SECONDS = 60 * 60
MAX_SCREEN = 81
MAX_FULL = 6
MAX_REFINE = 8
C_VALUES = (1, 10, 30)
GAMMA_VALUES = ("scale", 1, 3)
WEIGHT_PROFILES = (
    (1, 1, 1, 0.1, 0.1, 0.1, 0.1),
    (2, 1, 1, 0.1, 0.1, 0.1, 0.1),
    (2, 1, 1, 0.3, 0.3, 0.3, 0.3),
)
GLOBAL_L2 = True
VARIANTS = ("rgb", "avg", "ntsc")
BLOCK_NAMES = ("rgb", "avg", "ntsc", "hog", "lbp", "hsv", "rgb_stats")
for folder in ("model", "fitur", "laporan"):
    (OUT / folder).mkdir(parents=True, exist_ok=True)
print("Run:", RUN_ID, "Seed:", SEED)

Run: cifar10_7fitur_v2 Seed: 30092026


## 2. Data dan split

Muat label training resmi melalui `tensorflow.keras.datasets.cifar10`. Official test tidak disentuh. Indeks development diverifikasi terhadap arsip Combination.

In [2]:
from tensorflow.keras.datasets import cifar10

(x_all, y_all), _ = cifar10.load_data()
y_all = y_all.ravel().astype(np.int64)
if x_all.shape != (50000, 32, 32, 3):
    raise ValueError(f"Bentuk training tidak sesuai: {x_all.shape}")
train_idx, val_idx = train_test_split(
    np.arange(50000), test_size=0.2, stratify=y_all, random_state=SEED
)
train_idx, val_idx = np.sort(train_idx), np.sort(val_idx)
split_hash = hashlib.sha256(train_idx.tobytes() + val_idx.tobytes()).hexdigest()
print("Development:", len(train_idx), len(val_idx), split_hash)

I0000 00:00:1791496438.778679  226217 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791496438.819788  226217 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791496439.780300  226217 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Development: 40000 10000 16004a1ac8a89ac92ba6280367a7fb7b284c3643abb1c9d10409d6739ef31305


## 3. Periksa manifest dan arsip fitur

Hash arsip, checkpoint, label, indeks, batas blok, dan versi preprocessing harus cocok sebelum pencarian. Hash ini menjadi bagian identitas ledger pencarian.

In [3]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Jalankan Combination terlebih dahulu: {MANIFEST_PATH}")
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
if (manifest["run_id"] != RUN_ID or manifest["source_run"] != SOURCE_RUN
        or manifest["split_hash"] != split_hash or manifest["seed"] != SEED
        or tuple(manifest["block_names"]) != BLOCK_NAMES
        or manifest["preprocessing_version"] != "v2-group-l2-1"):
    raise ValueError("Manifest Combination tidak cocok")
for variant, expected_hash in zip(VARIANTS, manifest["model_hashes"]):
    path = ROOT / "outputs" / SOURCE_RUN / variant / "model" / "cnn.keras"
    if sha256_file(path) != expected_hash:
        raise ValueError(f"Checkpoint CNN {variant} berubah")
archive_paths = {}
for rep, dim in (("A", 2173), ("B", 2173), ("C", 1405)):
    path = OUT / "fitur" / f"train_combined_{rep}.npz"
    if not path.exists() or sha256_file(path) != manifest["archive_sha256"][rep]:
        raise ValueError(f"Arsip kandidat {rep} hilang atau hash berbeda")
    with np.load(path, allow_pickle=False) as archive:
        checks = [str(archive["run_id"]) == RUN_ID,
                  str(archive["representation"]) == rep,
                  str(archive["source_run"]) == SOURCE_RUN,
                  str(archive["split_hash"]) == split_hash,
                  str(archive["preprocessing_version"]) == "v2-group-l2-1",
                  tuple(archive["block_names"].tolist()) == BLOCK_NAMES,
                  np.array_equal(archive["boundaries"], manifest["boundaries"][rep]),
                  np.array_equal(archive["model_hashes"], manifest["model_hashes"]),
                  np.array_equal(archive["indices"], np.arange(50000)),
                  np.array_equal(archive["y"], y_all),
                  np.array_equal(archive["train_idx"], train_idx),
                  np.array_equal(archive["val_idx"], val_idx),
                  archive["features"].shape == (50000, dim),
                  np.isfinite(archive["features"]).all()]
        if not all(checks):
            raise ValueError(f"Metadata atau nilai fitur kandidat {rep} tidak cocok")
    archive_paths[rep] = path
print("Tiga arsip valid:", {rep: manifest["boundaries"][rep][-1] for rep in archive_paths})

Tiga arsip valid: {'A': 2173, 'B': 2173, 'C': 1405}


## 4. Definisi preprocessing per kelompok

Setiap blok di-StandardScaler berdasarkan fold training, lalu dinormalisasi L2 sendiri. Bobot mengatur kontribusi tujuh blok; L2 gabungan mempertahankan perbandingan bobot tersebut. Jika semua blok bernorm 1, dimensi 512 dan dimensi 9 mulai dengan norma yang sebanding. Jarak kuadrat tiap blok kemudian berkontribusi sebanding dengan kuadrat bobotnya.

In [4]:
class GroupPreprocessor(TransformerMixin, BaseEstimator):
    def __init__(self, boundaries, weights, global_l2=True):
        self.boundaries = boundaries
        self.weights = weights
        self.global_l2 = global_l2

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=np.float32)
        if len(self.boundaries) != 8 or len(self.weights) != 7:
            raise ValueError("Harus ada tujuh blok dan tujuh bobot")
        if X.shape[1] != self.boundaries[-1] or self.boundaries[0] != 0:
            raise ValueError("Batas blok tidak cocok dengan fitur")
        if any(np.diff(self.boundaries) <= 0) or any(w <= 0 for w in self.weights):
            raise ValueError("Batas blok atau bobot tidak valid")
        self.scalers_ = []
        for left, right in zip(self.boundaries[:-1], self.boundaries[1:]):
            scaler = StandardScaler()
            scaler.fit(X[:, left:right])
            self.scalers_.append(scaler)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=np.float32)
        if X.shape[1] != self.boundaries[-1]:
            raise ValueError("Dimensi fitur berbeda dari saat fitting")
        parts = []
        for scaler, left, right, weight in zip(
                self.scalers_, self.boundaries[:-1], self.boundaries[1:], self.weights):
            scaled = scaler.transform(X[:, left:right]).astype(np.float32)
            parts.append(Normalizer().transform(scaled) * np.float32(weight))
        result = np.concatenate(parts, axis=1)
        if self.global_l2:
            result = Normalizer().transform(result)
        if not np.isfinite(result).all():
            raise ValueError("Preprocessing menghasilkan NaN atau inf")
        return result

## 5. Subset screening dan identitas eksperimen

Ambil 12.000 dari 40.000 training serta 3.000 dari 10.000 validation secara stratified. Id sampel dan fingerprint konfigurasi disimpan agar hasil resume tidak mencampur arsip atau preprocessing berbeda.

In [5]:
screen_train_idx, _ = train_test_split(
    train_idx, train_size=12000, stratify=y_all[train_idx], random_state=SEED
)
screen_val_idx, _ = train_test_split(
    val_idx, train_size=3000, stratify=y_all[val_idx], random_state=SEED
)
screen_train_idx = np.sort(screen_train_idx)
screen_val_idx = np.sort(screen_val_idx)
subset_hash = hashlib.sha256(
    screen_train_idx.tobytes() + screen_val_idx.tobytes()
).hexdigest()
identity = {
    "run_id": RUN_ID, "source_run": SOURCE_RUN, "seed": SEED,
    "split_hash": split_hash, "subset_hash": subset_hash,
    "model_hashes": manifest["model_hashes"],
    "archive_sha256": manifest["archive_sha256"],
    "preprocessing_version": manifest["preprocessing_version"],
    "weights": WEIGHT_PROFILES, "global_l2": GLOBAL_L2,
    "initial_C": C_VALUES, "initial_gamma": GAMMA_VALUES,
    "sklearn_version": __import__("sklearn").__version__,
}
identity_sha = hashlib.sha256(
    json.dumps(identity, sort_keys=True).encode("utf-8")
).hexdigest()
print("Subset:", len(screen_train_idx), len(screen_val_idx), subset_hash)
print("Fingerprint:", identity_sha)

Subset: 12000 3000 761213e169f7f9a63734dd9e27b87541d828f37dc5bf04b68da422c9742f08f8
Fingerprint: f19e28f630afc506d257fbb9b8e967690786186501e98e320d07992a65ed591d


## 6. Diagnostik norma dan jarak

Pakai hanya 12.000 training. Ukur norma tiap blok sebelum dan sesudah scaling serta setelah bobot. Sampel 20.000 pasang jarak kuadrat, terpisah untuk kelas sama/berbeda. Persentil jarak dan nilai kernel `exp(-gamma × jarak²)` membantu menilai gamma sebelum GridSearchCV.


In [6]:
feature_cache = {}
def load_features(rep):
    if rep not in feature_cache:
        with np.load(archive_paths[rep], allow_pickle=False) as archive:
            feature_cache[rep] = archive["features"].astype(np.float32)
    return feature_cache[rep]

rng = np.random.default_rng(SEED)
diagnostics = {}
for rep, profile_id in (("B", 0), ("B", 1), ("C", 2), ("A", 0)):
    X = load_features(rep)[screen_train_idx]
    pre = GroupPreprocessor(tuple(manifest["boundaries"][rep]),
                            WEIGHT_PROFILES[profile_id], GLOBAL_L2)
    transformed = pre.fit_transform(X)
    left = rng.integers(0, len(X), 20000)
    right = rng.integers(0, len(X), 20000)
    distance2 = np.sum((transformed[left] - transformed[right]) ** 2, axis=1)
    gamma_scale = 1.0 / (transformed.shape[1] * transformed.var())
    positive = distance2[distance2 > 0]
    if len(positive) == 0:
        raise ValueError(f"Semua jarak kandidat {rep}-P{profile_id + 1} nol")
    median = float(np.median(positive))
    same_class = y_all[screen_train_idx][left] == y_all[screen_train_idx][right]
    candidates = {"scale": gamma_scale, "1": 1.0, "3": 3.0,
                  "median_inverse": 1.0 / median, "3_median_inverse": 3.0 / median}
    key = f"{rep}-P{profile_id + 1}"
    diagnostics[key] = {
        "distance2_percentiles": np.percentile(distance2, [5, 25, 50, 75, 95]).tolist(),
        "median_distance2": median,
        "same_class_median_distance2": float(np.median(distance2[same_class])),
        "different_class_median_distance2": float(np.median(distance2[~same_class])),
        "gamma_scale": float(gamma_scale),
        "raw_block_norm_median": [float(np.median(np.linalg.norm(
            X[:, a:b], axis=1)))
            for a, b in zip(manifest["boundaries"][rep][:-1],
                            manifest["boundaries"][rep][1:])],
        "standardized_block_norm_median": [float(np.median(np.linalg.norm(
            scaler.transform(X[:, a:b]), axis=1)))
            for scaler, a, b in zip(pre.scalers_,
                                    manifest["boundaries"][rep][:-1],
                                    manifest["boundaries"][rep][1:])],
        "weighted_block_norm_median": [float(np.median(np.linalg.norm(
            transformed[:, a:b], axis=1)))
            for a, b in zip(manifest["boundaries"][rep][:-1],
                            manifest["boundaries"][rep][1:])],
        "kernel_median": {name: float(np.median(np.exp(-gamma * distance2)))
                          for name, gamma in candidates.items()},
        "kernel_extreme_fraction": {
            name: float(np.mean((np.exp(-gamma * distance2) > 0.99)
                                | (np.exp(-gamma * distance2) < 1e-6)))
            for name, gamma in candidates.items()
        },
    }
    print(key, diagnostics[key])
(OUT / "laporan" / "distance_diagnostics.json").write_text(
    json.dumps({"identity_sha256": identity_sha, "results": diagnostics}, indent=2),
    encoding="utf-8")

B-P1 {'distance2_percentiles': [0.3759454429149628, 1.9811161756515503, 2.1802525520324707, 2.3065508008003235, 2.4332053899765014], 'median_distance2': 2.1802780628204346, 'same_class_median_distance2': 0.371757447719574, 'different_class_median_distance2': 2.206540822982788, 'gamma_scale': 1.0008943377101847, 'raw_block_norm_median': [16.751041412353516, 16.549837112426758, 16.775196075439453, 3.0, 0.14494635164737701, 0.7671766877174377, 1.3130825757980347], 'standardized_block_norm_median': [11.920750617980957, 12.087634086608887, 12.09044075012207, 17.696544647216797, 15.160467147827148, 5.843745231628418, 2.4799551963806152], 'weighted_block_norm_median': [0.5735393166542053, 0.5735393166542053, 0.5735393166542053, 0.05735393241047859, 0.05735393241047859, 0.05735393241047859, 0.05735393241047859], 'kernel_median': {'scale': 0.11279284954071045, '1': 0.11301298439502716, '3': 0.001443394459784031, 'median_inverse': 0.36788374185562134, '3_median_inverse': 0.049788810312747955}, '

6254

## 7. Ledger dan anggaran

Setiap kandidat mempunyai status `running` atau `complete`. Saat kernel terhenti di tengah kandidat, batch satu kandidat itu harus diulang. Durasi sejak status `running` dicatat saat resume secara konservatif. Fingerprint mencegah penggabungan hasil dari arsip atau parameter berbeda. Anggaran tidak direset dengan mengganti ID lama.

In [7]:
def atomic_json(path, payload):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    os.replace(temporary, path)

if LEDGER_PATH.exists():
    ledger = json.loads(LEDGER_PATH.read_text(encoding="utf-8"))
    if ledger["identity_sha256"] != identity_sha:
        raise ValueError("Ledger v2 berasal dari arsip atau konfigurasi lain")
    for entry in ledger["candidates"].values():
        if entry["status"] == "running":
            lost = max(0.0, time.time() - entry["started_unix"])
            ledger["spent_seconds"] += lost
            entry["status"] = "interrupted"
            entry["interrupted_seconds"] = lost
    atomic_json(LEDGER_PATH, ledger)
else:
    ledger = {
        "identity_sha256": identity_sha, "identity": identity,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "spent_seconds": float(manifest["combination_seconds"]),
        "candidates": {},
        "screen_train_idx": screen_train_idx.tolist(),
        "screen_val_idx": screen_val_idx.tolist(),
        "budget_seconds": BUDGET_SECONDS,
    }
    atomic_json(LEDGER_PATH, ledger)
SEARCH_LOCKED = (OUT / "laporan" / "selection.json").exists()
print("Waktu tercatat:", round(ledger["spent_seconds"] / 60, 1), "menit")
print("Seleksi terkunci:", SEARCH_LOCKED)

Waktu tercatat: 2.8 menit
Seleksi terkunci: False


## 8. GridSearchCV per kandidat

Satu pemanggilan GridSearchCV menjalankan tepat satu kombinasi parameter pada `PredefinedSplit`; scaler di-fit hanya pada training fold. Skor, durasi, dan status ditulis setelah tiap kandidat. Kandidat selesai dilewati saat resume.

In [8]:
def candidate_key(rep, profile_id, C, gamma, phase):
    return json.dumps({"phase": phase, "representation": rep,
                       "profile_id": profile_id, "C": C, "gamma": gamma},
                      sort_keys=True)


def run_candidate(rep, profile_id, C, gamma, phase, train_ids, validation_ids):
    key = candidate_key(rep, profile_id, C, gamma, phase)
    prior = ledger["candidates"].get(key)
    if prior and prior["status"] == "complete":
        print("Resume: lewati", key, "score", prior["score"])
        return prior
    if SEARCH_LOCKED:
        raise RuntimeError("Seleksi sudah dibekukan; kandidat baru tidak boleh dijalankan")
    X = load_features(rep)
    ids = np.concatenate([train_ids, validation_ids])
    fold = np.concatenate([np.full(len(train_ids), -1, np.int8),
                           np.zeros(len(validation_ids), np.int8)])
    pipe = Pipeline([
        ("groups", GroupPreprocessor(tuple(manifest["boundaries"][rep]),
                                     WEIGHT_PROFILES[profile_id], GLOBAL_L2)),
        ("svc", SVC(kernel="rbf", cache_size=512, random_state=SEED, verbose=True)),
    ])
    grid = GridSearchCV(
        pipe, {"svc__C": [C], "svc__gamma": [gamma]},
        cv=PredefinedSplit(fold), scoring="accuracy", refit=False,
        n_jobs=1, error_score="raise", verbose=1,
    )
    started = time.time()
    ledger["candidates"][key] = {
        "status": "running", "started_unix": started,
        "phase": phase, "representation": rep, "profile_id": profile_id,
        "C": C, "gamma": gamma,
        "training_count": len(train_ids), "validation_count": len(validation_ids),
    }
    atomic_json(LEDGER_PATH, ledger)
    print("Mulai:", key, flush=True)
    grid.fit(X[ids], y_all[ids])
    duration = time.time() - started
    score = float(grid.cv_results_["mean_test_score"][0])
    entry = ledger["candidates"][key]
    entry.update(status="complete", score=score, seconds=duration,
                 finished_utc=datetime.now(timezone.utc).isoformat())
    ledger["spent_seconds"] += duration
    atomic_json(LEDGER_PATH, ledger)
    print("Selesai:", key, "score", score, "detik", round(duration, 1), flush=True)
    return entry

## 9. Benchmark tiga konfigurasi

Tiga kandidat mewakili B dengan dua profil serta C dengan profil ketiga. Waktu screening menjadi dasar proyeksi biaya 40.000/10.000. Jika hasil pengukuran jauh lebih mahal, jumlah kandidat berikutnya dikurangi.

In [9]:
benchmarks = [("B", 0, 1, "scale"), ("B", 1, 10, 1), ("C", 2, 30, 3)]
for rep, profile_id, C, gamma in benchmarks:
    if SEARCH_LOCKED:
        print("Seleksi terkunci; benchmark dilewati")
        break
    if ledger["spent_seconds"] + FINAL_RESERVE_SECONDS >= BUDGET_SECONDS:
        print("Benchmark dihentikan untuk menyisakan waktu final")
        break
    run_candidate(rep, profile_id, C, gamma, "screen",
                  screen_train_idx, screen_val_idx)
benchmark_entries = [ledger["candidates"][candidate_key(*row, "screen")]
                     for row in benchmarks
                     if candidate_key(*row, "screen") in ledger["candidates"]
                     and ledger["candidates"][candidate_key(*row, "screen")]["status"] == "complete"]
if not benchmark_entries:
    raise RuntimeError("Tidak ada benchmark selesai; anggaran atau kernel perlu diperiksa")
screen_estimate = float(np.median([e["seconds"] for e in benchmark_entries]))
full_estimate = screen_estimate * (40000 / 12000) ** 1.8
print("Estimasi per kandidat: screening", round(screen_estimate, 1),
      "detik; full", round(full_estimate, 1), "detik")

Mulai: {"C": 1, "gamma": "scale", "phase": "screen", "profile_id": 0, "representation": "B"}
Fitting 1 folds for each of 1 candidates, totalling 1 fits
[LibSVM]*
optimization finished, #iter = 101
obj = -3.572875, rho = -0.014992
nSV = 46, nBSV = 0
*
optimization finished, #iter = 85
obj = -4.911112, rho = 0.037001
nSV = 39, nBSV = 2
*
optimization finished, #iter = 89
obj = -3.781403, rho = -0.004905
nSV = 41, nBSV = 0
*
optimization finished, #iter = 74
obj = -3.825650, rho = -0.010658
nSV = 37, nBSV = 2
*
optimization finished, #iter = 81
obj = -3.252759, rho = -0.071282
nSV = 39, nBSV = 0
*
optimization finished, #iter = 73
obj = -3.592550, rho = 0.003069
nSV = 36, nBSV = 0
*
optimization finished, #iter = 88
obj = -3.843388, rho = 0.042294
nSV = 39, nBSV = 0
*
optimization finished, #iter = 84
obj = -4.052948, rho = -0.011510
nSV = 40, nBSV = 0
*
optimization finished, #iter = 85
obj = -3.895777, rho = 0.012502
nSV = 37, nBSV = 0
*
optimization finished, #iter = 85
obj = -3.618462

## 10. Pilot validation penuh

Kandidat B terbaik dari benchmark diuji pada 40.000/10.000 sedini mungkin. Durasi nyata pilot mengganti proyeksi full bila tersedia, agar screening tidak menghabiskan waktu untuk evaluasi penuh.

In [10]:
b_benchmarks = [e for e in benchmark_entries if e["representation"] == "B"]
if not b_benchmarks:
    raise RuntimeError("Benchmark B belum selesai")
pilot = max(b_benchmarks, key=lambda e: (e["score"], -e["seconds"]))
if (not SEARCH_LOCKED and ledger["spent_seconds"] + full_estimate
        + FINAL_RESERVE_SECONDS <= BUDGET_SECONDS):
    pilot_full = run_candidate("B", pilot["profile_id"], pilot["C"],
                               pilot["gamma"], "full", train_idx, val_idx)
    full_estimate = pilot_full["seconds"]
else:
    print("Pilot full dilewati karena anggaran; coba lanjutkan dengan anggaran yang tersedia")
print("Perkiraan fit 50K:", round(full_estimate * 1.5 / 60, 1), "menit")

Mulai: {"C": 10, "gamma": 1, "phase": "full", "profile_id": 1, "representation": "B"}
Fitting 1 folds for each of 1 candidates, totalling 1 fits
[LibSVM]*
optimization finished, #iter = 115
obj = -4.762387, rho = -0.039047
nSV = 52, nBSV = 0
*
optimization finished, #iter = 94
obj = -6.626965, rho = -0.006475
nSV = 38, nBSV = 0
*
optimization finished, #iter = 112
obj = -5.388395, rho = 0.005455
nSV = 49, nBSV = 0
*
optimization finished, #iter = 93
obj = -5.033142, rho = -0.003727
nSV = 41, nBSV = 0
*
optimization finished, #iter = 111
obj = -4.056459, rho = -0.114367
nSV = 46, nBSV = 0
*
optimization finished, #iter = 120
obj = -4.484987, rho = -0.036732
nSV = 50, nBSV = 0
*
optimization finished, #iter = 126
obj = -4.779545, rho = -0.049784
nSV = 51, nBSV = 0
*
optimization finished, #iter = 118
obj = -5.353290, rho = -0.038779
nSV = 49, nBSV = 0
*
optimization finished, #iter = 95
obj = -5.317907, rho = 0.008099
nSV = 44, nBSV = 0
*
optimization finished, #iter = 73
obj = -4.423229

## 11. Screening adaptif dengan GridSearchCV

Urutan B, A, C mengutamakan flip Dense sambil menjaga pembanding. Batas 81 kombinasi adalah maksimum; pencarian berhenti ketika 40 menit screening atau cadangan full dan final terancam. Gamma awal dipakai jika median kernel tidak degenerat; jika semuanya degenerat, dua nilai numerik berbasis median jarak menggantikan 1 dan 3.

In [11]:
def numeric_gamma_grid():
    all_extreme = all(
        all(value["kernel_extreme_fraction"][key] > 0.99
            for key in ("scale", "1", "3"))
        for value in diagnostics.values()
    )
    if all_extreme:
        median = float(np.median([value["median_distance2"]
                                  for value in diagnostics.values()]))
        print("Kernel awal degenerat; gamma numerik berdasarkan median jarak", median)
        return ("scale", round(1.0 / median, 8), round(3.0 / median, 8))
    return GAMMA_VALUES

active_gamma = numeric_gamma_grid()
screen_plan = [(rep, profile_id, C, gamma)
               for profile_id in range(3)
               for C in C_VALUES
               for gamma in active_gamma
               for rep in ("B", "A", "C")]
for rep, profile_id, C, gamma in screen_plan[:MAX_SCREEN]:
    if SEARCH_LOCKED:
        print("Seleksi terkunci; screening dilewati")
        break
    complete_screen = [e for e in ledger["candidates"].values()
                       if e["phase"] == "screen" and e["status"] == "complete"]
    screen_seconds = sum(e["seconds"] for e in complete_screen)
    if (screen_seconds + screen_estimate > SCREEN_BUDGET_SECONDS
            or ledger["spent_seconds"] + screen_estimate + full_estimate
            + FINAL_RESERVE_SECONDS > BUDGET_SECONDS):
        print("Screening berhenti karena cadangan waktu; selesai", len(complete_screen))
        break
    run_candidate(rep, profile_id, C, gamma, "screen",
                  screen_train_idx, screen_val_idx)
print("Kandidat screening selesai:", sum(
    e["phase"] == "screen" and e["status"] == "complete"
    for e in ledger["candidates"].values()))

Resume: lewati {"C": 1, "gamma": "scale", "phase": "screen", "profile_id": 0, "representation": "B"} score 0.9373333333333334
Mulai: {"C": 1, "gamma": "scale", "phase": "screen", "profile_id": 0, "representation": "A"}
Fitting 1 folds for each of 1 candidates, totalling 1 fits
[LibSVM]*
optimization finished, #iter = 92
obj = -3.822262, rho = -0.028754
nSV = 46, nBSV = 0
*
optimization finished, #iter = 87
obj = -5.571965, rho = 0.024323
nSV = 44, nBSV = 2
*
optimization finished, #iter = 100
obj = -4.028899, rho = -0.023061
nSV = 49, nBSV = 0
*
optimization finished, #iter = 73
obj = -4.081842, rho = -0.035148
nSV = 37, nBSV = 1
*
optimization finished, #iter = 84
obj = -3.527866, rho = -0.071929
nSV = 42, nBSV = 0
*
optimization finished, #iter = 89
obj = -3.762734, rho = -0.002210
nSV = 42, nBSV = 0
*
optimization finished, #iter = 80
obj = -4.067825, rho = 0.030491
nSV = 40, nBSV = 0
*
optimization finished, #iter = 96
obj = -4.460460, rho = -0.008549
nSV = 43, nBSV = 0
*
optimizat

## 12. Validation penuh kandidat teratas

Pilih kandidat screening terbaik per representasi, lalu sisanya menurut skor, maksimum enam konfigurasi. Semua diukur pada split 40.000/10.000; seleksi akhir tidak memakai official test.

In [12]:
screened = [e for e in ledger["candidates"].values()
            if e["phase"] == "screen" and e["status"] == "complete"]
ranked_screen = sorted(screened, key=lambda e: (-e["score"],
                     {"B": 0, "A": 1, "C": 2}[e["representation"]], e["profile_id"]))
full_plan = []
for rep in ("B", "A", "C"):
    match = next((e for e in ranked_screen if e["representation"] == rep), None)
    if match is not None:
        full_plan.append(match)
for entry in ranked_screen:
    signature = (entry["representation"], entry["profile_id"], entry["C"], entry["gamma"])
    if all(signature != (e["representation"], e["profile_id"], e["C"], e["gamma"])
           for e in full_plan):
        full_plan.append(entry)
full_plan = full_plan[:MAX_FULL]
for entry in full_plan:
    if SEARCH_LOCKED:
        print("Seleksi terkunci; validation penuh dilewati")
        break
    complete_full = [e for e in ledger["candidates"].values()
                     if e["phase"] == "full" and e["status"] == "complete"]
    if (sum(e["seconds"] for e in complete_full) + full_estimate > FULL_BUDGET_SECONDS
            or ledger["spent_seconds"] + full_estimate + FINAL_RESERVE_SECONDS
            > BUDGET_SECONDS):
        print("Validation penuh berhenti untuk menjaga anggaran final")
        break
    result = run_candidate(entry["representation"], entry["profile_id"],
                           entry["C"], entry["gamma"], "full", train_idx, val_idx)
    full_estimate = float(np.median([e["seconds"] for e in ledger["candidates"].values()
                                    if e["phase"] == "full" and e["status"] == "complete"]))
print("Full validation selesai:", sum(
    e["phase"] == "full" and e["status"] == "complete"
    for e in ledger["candidates"].values()))

Resume: lewati {"C": 10, "gamma": 1, "phase": "full", "profile_id": 1, "representation": "B"} score 0.9387
Mulai: {"C": 1, "gamma": "scale", "phase": "full", "profile_id": 1, "representation": "A"}
Fitting 1 folds for each of 1 candidates, totalling 1 fits
[LibSVM]*
optimization finished, #iter = 88
obj = -5.243181, rho = -0.044781
nSV = 40, nBSV = 1
*
optimization finished, #iter = 100
obj = -8.108729, rho = -0.049007
nSV = 46, nBSV = 3
*
optimization finished, #iter = 114
obj = -6.224314, rho = 0.017831
nSV = 47, nBSV = 1
*
optimization finished, #iter = 109
obj = -5.541836, rho = -0.015368
nSV = 48, nBSV = 1
*
optimization finished, #iter = 136
obj = -4.267249, rho = -0.123522
nSV = 55, nBSV = 0
*
optimization finished, #iter = 120
obj = -4.950319, rho = -0.049262
nSV = 59, nBSV = 0
*
optimization finished, #iter = 121
obj = -5.181579, rho = -0.049848
nSV = 55, nBSV = 1
*
optimization finished, #iter = 95
obj = -6.192180, rho = -0.035530
nSV = 49, nBSV = 3
*
optimization finished, #

## 13. Refinement opsional dengan GridSearchCV

Jika waktu cukup, uji dua C di sekitar kandidat terbaik dan dua gamma berbasis median jarak. Maksimum delapan kandidat pada validation penuh. Semua hasil tetap ditulis per kandidat ke ledger.

In [13]:
full_results = [e for e in ledger["candidates"].values()
                if e["phase"] in ("full", "refine") and e["status"] == "complete"]
if full_results:
    leader = max(full_results, key=lambda e: e["score"])
    diag = diagnostics.get(f'{leader["representation"]}-P{leader["profile_id"] + 1}')
    if diag is None:
        diag = next(iter(diagnostics.values()))
    median = diag["median_distance2"]
    refine_pairs = [(max(0.1, leader["C"] / 2), leader["gamma"]),
                    (leader["C"] * 2, leader["gamma"]),
                    (leader["C"], round(1.0 / median, 8)),
                    (leader["C"], round(3.0 / median, 8))]
    refine_pairs = list(dict.fromkeys(refine_pairs))[:MAX_REFINE]
    for C, gamma in refine_pairs:
        if SEARCH_LOCKED:
            print("Seleksi terkunci; refinement dilewati")
            break
        prior_refine = [e for e in ledger["candidates"].values()
                        if e["phase"] == "refine" and e["status"] == "complete"]
        if (sum(e["seconds"] for e in prior_refine) + full_estimate
                > REFINE_BUDGET_SECONDS
                or ledger["spent_seconds"] + full_estimate + FINAL_RESERVE_SECONDS
                > BUDGET_SECONDS):
            print("Refinement dilewati: anggaran tersisa dipakai untuk final")
            break
        run_candidate(leader["representation"], leader["profile_id"],
                      C, gamma, "refine", train_idx, val_idx)

Mulai: {"C": 5.0, "gamma": 1, "phase": "refine", "profile_id": 1, "representation": "B"}
Fitting 1 folds for each of 1 candidates, totalling 1 fits
[LibSVM]*
optimization finished, #iter = 115
obj = -4.762387, rho = -0.039047
nSV = 52, nBSV = 0
*
optimization finished, #iter = 94
obj = -6.626965, rho = -0.006475
nSV = 38, nBSV = 0
*
optimization finished, #iter = 112
obj = -5.388395, rho = 0.005455
nSV = 49, nBSV = 0
*
optimization finished, #iter = 93
obj = -5.033142, rho = -0.003727
nSV = 41, nBSV = 0
*
optimization finished, #iter = 111
obj = -4.056459, rho = -0.114367
nSV = 46, nBSV = 0
*
optimization finished, #iter = 120
obj = -4.484987, rho = -0.036732
nSV = 50, nBSV = 0
*
optimization finished, #iter = 126
obj = -4.779545, rho = -0.049784
nSV = 51, nBSV = 0
*
optimization finished, #iter = 118
obj = -5.353290, rho = -0.038779
nSV = 49, nBSV = 0
*
optimization finished, #iter = 95
obj = -5.317907, rho = 0.008099
nSV = 44, nBSV = 0
*
optimization finished, #iter = 73
obj = -4.423

## 14. Bekukan konfigurasi terpilih

Skor tertinggi pada 10.000 validation menentukan representasi, bobot, C, dan gamma. Pembanding v1 93,35% dihitung pada split yang sama. File seleksi membekukan parameter sebelum final fitting dan sebelum official test.

In [14]:
full_results = [e for e in ledger["candidates"].values()
                if e["phase"] in ("full", "refine") and e["status"] == "complete"]
if not full_results:
    raise RuntimeError("Belum ada kandidat pada 40K/10K; lanjutkan notebook saat anggaran memungkinkan")
winner = max(full_results, key=lambda e: (e["score"], e["representation"] == "B"))
v1_report_path = ROOT / "outputs" / SOURCE_RUN / "fusion" / "laporan" / "fusion_grid.json"
v1_report = json.loads(v1_report_path.read_text(encoding="utf-8"))
if v1_report["split_hash"] != split_hash:
    raise ValueError("Baseline v1 memakai split berbeda")
v1_validation = float(v1_report["fusion_final_validation_accuracy"])
selection = {
    "run_id": RUN_ID, "identity_sha256": identity_sha,
    "representation": winner["representation"],
    "profile_id": winner["profile_id"],
    "weights": WEIGHT_PROFILES[winner["profile_id"]],
    "C": winner["C"], "gamma": winner["gamma"],
    "validation_accuracy": winner["score"],
    "v1_validation_accuracy": v1_validation,
    "v1_three_embeddings_validation_accuracy": float(
        v1_report["baseline_final_validation_accuracy"]),
    "validation_delta": winner["score"] - v1_validation,
    "archive_sha256": manifest["archive_sha256"][winner["representation"]],
    "model_hashes": manifest["model_hashes"],
    "split_hash": split_hash,
    "boundaries": manifest["boundaries"][winner["representation"]],
    "preprocessing_version": manifest["preprocessing_version"],
    "official_test_status": "belum dievaluasi pada v2; v1 sudah pernah terpapar",
}
selection_path = OUT / "laporan" / "selection.json"
if selection_path.exists():
    previous = json.loads(selection_path.read_text(encoding="utf-8"))
    if previous != json.loads(json.dumps(selection)):
        raise ValueError("Seleksi sudah dibekukan dan berbeda; periksa ledger")
else:
    atomic_json(selection_path, selection)
print("Terpilih:", selection)

Terpilih: {'run_id': 'cifar10_7fitur_v2', 'identity_sha256': 'f19e28f630afc506d257fbb9b8e967690786186501e98e320d07992a65ed591d', 'representation': 'B', 'profile_id': 1, 'weights': (2, 1, 1, 0.1, 0.1, 0.1, 0.1), 'C': 10, 'gamma': 0.45854146, 'validation_accuracy': 0.939, 'v1_validation_accuracy': 0.9335, 'v1_three_embeddings_validation_accuracy': 0.933, 'validation_delta': 0.005499999999999949, 'archive_sha256': '913e80a3acc3508ef3fab0750b0df51bfbf885446149ca03766aa3027fa57fc8', 'model_hashes': ['f7ee1fafbaa536b419c50eb6a1f749ea6fe7051dc4fab6825ec19a904e6ab912', 'efc5598b6789db22d062c076d0641e1afee43076a87312ccc70cdb55cb7d4010', '8154382b1f8e64db3ccd7e3af5d38dd2a912b5f681b37973fe8abf5ffb439d13'], 'split_hash': '16004a1ac8a89ac92ba6280367a7fb7b284c3643abb1c9d10409d6739ef31305', 'boundaries': [0, 512, 1024, 1536, 1860, 2116, 2164, 2173], 'preprocessing_version': 'v2-group-l2-1', 'official_test_status': 'belum dievaluasi pada v2; v1 sudah pernah terpapar'}


## 15. Fit final pada 50.000 training

Setelah seleksi dibekukan, preprocessing dan SVM dilatih ulang pada semua 50.000 official training. Validation kini boleh bergabung karena tidak dipakai lagi untuk memilih parameter. Jika model yang kompatibel sudah tersedia, fitting dilewati.


In [15]:
final_model_path = OUT / "model" / "fusion_svm_final.joblib"
selection_sha = sha256_file(selection_path)
final_needs_save = not final_model_path.exists()
if not final_needs_save:
    payload = joblib.load(final_model_path)
    if (payload.get("selection_sha256") != selection_sha
            or payload.get("archive_sha256") != selection["archive_sha256"]
            or payload.get("training_count") != 50000):
        raise ValueError("Model final yang ada tidak cocok dengan seleksi")
    print("Model final valid, fitting dilewati:", final_model_path)
else:
    X_final = load_features(selection["representation"])
    final_pipeline = Pipeline([
        ("groups", GroupPreprocessor(tuple(selection["boundaries"]),
                                     tuple(selection["weights"]), GLOBAL_L2)),
        ("svc", SVC(kernel="rbf", C=selection["C"], gamma=selection["gamma"],
                    cache_size=512, random_state=SEED, verbose=True)),
    ])
    started = time.perf_counter()
    final_pipeline.fit(X_final, y_all)
    final_seconds = time.perf_counter() - started
    payload = {
        "pipeline": final_pipeline, "run_id": RUN_ID,
        "selection_sha256": selection_sha,
        "archive_sha256": selection["archive_sha256"],
        "representation": selection["representation"],
        "boundaries": selection["boundaries"],
        "model_hashes": selection["model_hashes"],
        "split_hash": split_hash,
        "preprocessing_version": selection["preprocessing_version"],
        "training_count": 50000,
        "fit_seconds": final_seconds,
    }


[LibSVM]*
optimization finished, #iter = 195
obj = -104.990043, rho = -0.293018
nSV = 57, nBSV = 5
*
optimization finished, #iter = 615
obj = -458.440572, rho = 0.080188
nSV = 142, nBSV = 31
*
optimization finished, #iter = 423
obj = -240.860903, rho = -0.095306
nSV = 96, nBSV = 12
*
optimization finished, #iter = 250
obj = -98.299872, rho = 0.016886
nSV = 66, nBSV = 3
*
optimization finished, #iter = 172
obj = -80.008821, rho = -0.062182
nSV = 50, nBSV = 1
*
optimization finished, #iter = 268
obj = -118.870283, rho = -0.091035
nSV = 72, nBSV = 4
*
optimization finished, #iter = 256
obj = -78.742718, rho = -0.211734
nSV = 56, nBSV = 1
*
optimization finished, #iter = 468
obj = -382.136636, rho = -0.159846
nSV = 115, nBSV = 26
*
optimization finished, #iter = 513
obj = -298.739681, rho = -0.159736
nSV = 119, nBSV = 17
*
optimization finished, #iter = 181
obj = -49.215253, rho = 0.353950
nSV = 49, nBSV = 0
*
optimization finished, #iter = 184
obj = -69.689697, rho = 0.401369
nSV = 55, nB

## 16. Simpan model final

Simpan pipeline dan provenance secara atomik. Durasi fitting ditambahkan ke ledger setelah model berhasil ditulis.


In [16]:
if final_needs_save:
    temporary = final_model_path.with_suffix(".joblib.tmp")
    joblib.dump(payload, temporary)
    os.replace(temporary, final_model_path)
    ledger["spent_seconds"] += final_seconds
    atomic_json(LEDGER_PATH, ledger)
    print("Final 50K selesai dalam", round(final_seconds / 60, 1), "menit")
    final_needs_save = False
else:
    print("Penyimpanan dilewati; model final sudah ada")


Final 50K selesai dalam 0.8 menit


## 17. Laporan waktu dan hasil development

Simpan seluruh skor, proyeksi, serta durasi aktual. Akurasi official test baru muncul setelah notebook Testing dijalankan; angka 95% belum dapat disimpulkan dari validation.


In [17]:
report = {
    "run_id": RUN_ID, "identity_sha256": identity_sha,
    "screen_subset": {"train": len(screen_train_idx), "validation": len(screen_val_idx),
                      "hash": subset_hash},
    "benchmark_screen_seconds_median": screen_estimate,
    "full_seconds_estimate_or_median": full_estimate,
    "candidate_counts": {
        phase: sum(e["phase"] == phase and e["status"] == "complete"
                   for e in ledger["candidates"].values())
        for phase in ("screen", "full", "refine")
    },
    "spent_seconds_recorded": ledger["spent_seconds"],
    "budget_seconds": BUDGET_SECONDS,
    "selection": selection,
    "final_fit_seconds": payload["fit_seconds"],
    "post_exposure_testing": True,
}
atomic_json(OUT / "laporan" / "training_report.json", report)
print("Validation tujuh fitur v2:", selection["validation_accuracy"])
print("Validation tujuh fitur v1:", selection["v1_validation_accuracy"])
print("Selisih:", selection["validation_delta"])
print("Waktu tercatat:", round(ledger["spent_seconds"] / 60, 1), "menit")

Validation tujuh fitur v2: 0.939
Validation tujuh fitur v1: 0.9335
Selisih: 0.005499999999999949
Waktu tercatat: 21.5 menit
